In [1]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.datasets import mnist 
import my_cnn
from scipy.ndimage import rotate


In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
y_train = np.eye(10)[y_train]
y_test= np.eye(10)[y_test]

In [3]:
angles = [-12, 0, 12]
new_x = []
new_y = []

for img, label in zip(x_train, y_train):
    for angle in angles:
        img_rot = rotate(img, angle, axes=(0,1), reshape=False, mode='constant', cval=0.0)
        new_x.append(img_rot)
        new_y.append(label)

x_train_augmented = np.array(new_x)
y_train_augmented = np.array(new_y)


In [4]:
test_model = my_cnn.Model()
test_model.add(my_cnn.layers.Input((28, 28, 1)))
test_model.add(my_cnn.layers.Rescaling(1./255))
test_model.add(my_cnn.layers.Convolution2D(32, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Convolution2D(64, 3, activation_func='relu'))
test_model.add(my_cnn.layers.MaxPooling2D(2))
test_model.add(my_cnn.layers.Flatten())
test_model.add(my_cnn.layers.Dense(64, activation_func='relu'))
test_model.add(my_cnn.layers.Dense(10, activation_func='softmax'))

In [5]:
test_model.compile(optimizer='momentum', loss='crossentropy', learning_rate=0.02)
test_model.fit(x_train_augmented, y_train_augmented, epochs=2, batch_size=64)

Epoch 1/2 | Batch 2813/2813 | Loss: 0.0905 | Acc: 0.9713
Epoch 2/2 | Batch 2813/2813 | Loss: 0.0327 | Acc: 0.9895


In [6]:
model_loss, model_accuracy = test_model.evaluate(x_test, y_test)

In [7]:
print(f"Loss: {model_loss}")
print(f"Accuracy: {model_accuracy*100:.2f}")

Loss: 0.024899599085017023
Accuracy: 99.15


In [8]:
predictions = test_model.predict(x_test)
decisions = np.argmax(predictions, axis=1)
y_test_classes = np.argmax(y_test, axis=1)

print(classification_report(y_test_classes, decisions))


              precision    recall  f1-score   support

           0       0.99      1.00      0.99       980
           1       1.00      1.00      1.00      1135
           2       0.99      0.99      0.99      1032
           3       0.99      1.00      0.99      1010
           4       0.98      1.00      0.99       982
           5       0.98      1.00      0.99       892
           6       1.00      0.98      0.99       958
           7       0.99      0.99      0.99      1028
           8       1.00      0.99      0.99       974
           9       0.99      0.98      0.99      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



In [9]:
cm = confusion_matrix(y_test_classes, decisions)
df_cm = pd.DataFrame(cm)
df_cm.index.name = "Real"
df_cm.columns.name = "Predicted"
df_cm 

Predicted,0,1,2,3,4,5,6,7,8,9
Real,,,,,,,,,,
0,976,0,0,0,0,1,2,1,0,0
1,0,1131,0,2,1,1,0,0,0,0
2,2,0,1026,0,1,0,0,2,1,0
3,0,0,0,1007,0,3,0,0,0,0
4,0,0,0,0,981,0,0,0,0,1
5,0,0,0,4,0,888,0,0,0,0
6,4,2,1,1,5,6,939,0,0,0
7,0,2,4,0,1,0,0,1017,0,4
8,1,0,3,0,1,2,0,1,963,3


In [10]:
test_model.save('../models/custom_cnn.pkl')

Model saved to: ../models/custom_cnn.pkl
